# Route Resilience — Kaggle-optimized road segmentation

This is a high-quality Kaggle training recipe for **DeepGlobe Roads**. It keeps
the project’s `p1_segment` API and swaps the weak spots in the original
notebook for a practical 16 GB-GPU workflow:

- MiT-B3 + U-Net at **512 px** (pretrained; stronger spatial detail than 256 px)
- road-aware crop sampling, geometric/colour augmentation, and occlusion training
- mixed precision, channels-last, pinned workers, gradient accumulation, clipping,
  discriminative learning rates, and warmup + cosine decay
- **full-resolution overlapping validation** and global IoU/threshold selection
- best/last resumable checkpoints, optional flip-TTA, and occlusion-recall reporting

## Kaggle checklist

1. Add the **DeepGlobe Road Extraction Dataset** (the `balraj98` dataset) with
   **+ Add Data**.
2. Turn on a GPU accelerator. The defaults are tuned to a 16 GB T4/P100 class GPU.
3. Turn on Internet for the first run so the pinned packages, source repository,
   and ImageNet encoder weights can be fetched. For an offline run, attach a Kaggle
   Dataset containing the already-cloned `Trace` repository and compatible wheels,
   then point `REPO_DIR` and `WHEEL_DIR` in the setup cells at those locations.
4. Run all cells. To keep a model, use **Save Version**; everything under
   `/kaggle/working/road_segmentation` becomes notebook output.

The default model is deliberately aggressive but should fit 16 GB. If the GPU reports
an out-of-memory error, change only `encoder` to `mit_b2` first; keep 512 px crops.


In [ ]:
# 1. Reproducible environment and pinned dependencies.
# On a normal Kaggle image this takes no action when compatible packages are present.
import importlib
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

REQUIRED = {
    "segmentation_models_pytorch": "segmentation-models-pytorch==0.3.4",
    "albumentations": "albumentations==2.0.8",
    "albucore": "albucore==0.0.24",
    "cv2": "opencv-python-headless==4.10.0.84",
}
PINNED_VERSIONS = {
    "segmentation-models-pytorch": "0.3.4",
    "albumentations": "2.0.8",
    "albucore": "0.0.24",
}
required_install = False
for module in REQUIRED:
    try:
        importlib.import_module(module)
    except ImportError:
        required_install = True
        break
if not required_install:
    for distribution, expected in PINNED_VERSIONS.items():
        try:
            if version(distribution) != expected:
                required_install = True
                break
        except PackageNotFoundError:
            required_install = True
            break

# Set this to an attached Kaggle Dataset of wheel files for an offline environment.
WHEEL_DIR = None
if required_install:
    cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if WHEEL_DIR:
        cmd += ["--no-index", "--find-links", WHEEL_DIR]
    cmd += list(REQUIRED.values())
    subprocess.check_call(cmd)

import albumentations as A
import cv2
import numpy as np
import segmentation_models_pytorch as smp
import torch

print("torch:", torch.__version__)
print("albumentations:", A.__version__, "| smp:", smp.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| memory (GiB):",
          round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


In [ ]:
# 2. Load the project source. The notebook only depends on the public P1 API.
import os
import shutil
from pathlib import Path

def detect_env():
    if os.path.exists("/kaggle") or os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    if "google.colab" in sys.modules or os.path.exists("/content"):
        return "colab"
    return "local"

ENV = detect_env()
REPO_URL = "https://github.com/Akshat-Tiwari69/Trace.git"
REPO_BRANCH = "dev"
REPO_DIR = "/kaggle/working/Trace" if ENV == "kaggle" else "/content/Trace" if ENV == "colab" else "."

# If you attached an offline source snapshot, set this to its directory before running.
OFFLINE_REPO_DIR = None
repo_path = Path(REPO_DIR)
if not (repo_path / "src" / "pipeline" / "p1_segment").exists():
    if OFFLINE_REPO_DIR:
        shutil.copytree(OFFLINE_REPO_DIR, repo_path, dirs_exist_ok=True)
    else:
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", REPO_BRANCH,
            REPO_URL, str(repo_path)
        ])

if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

from src.pipeline.p1_segment.model import build_model
from src.pipeline.p1_segment.losses import DiceBCELoss
from src.pipeline.p1_segment.metrics import occlusion_recall

print("environment:", ENV, "| source:", repo_path.resolve())


In [ ]:
# 3. Configuration, deterministic split, and input discovery.
import gc
import math
import random
from collections import defaultdict

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Speed settings are safe for this fixed-size training workload. Do not set
# torch.use_deterministic_algorithms(True): it is substantially slower here.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

CONFIG = {
    # Architecture / resolution. mit_b3 + 512 is the strongest reliable
    # default on a 16 GB Kaggle GPU. For OOM: change encoder -> mit_b2.
    "encoder": "mit_b3",
    "image_size": 512,
    "batch_size": 2,
    "grad_accum": 4,              # effective batch = 8 without a memory spike

    # Training budget. 30 epochs gives every 1024 px source image many
    # distinct 512 px views while remaining a practical single-session budget.
    "epochs": 30,
    "warmup_epochs": 2,
    "base_lr": 2.0e-4,
    "encoder_lr_scale": 0.35,
    "weight_decay": 1.0e-4,
    "min_lr_ratio": 0.03,
    "max_grad_norm": 1.0,

    # Data / crop balance. Roads occupy few pixels, so mostly request a
    # crop containing at least a trace of road while retaining background.
    "crops_per_image": 1,
    "positive_crop_probability": 0.72,
    "min_positive_fraction": 0.002,
    "max_crop_attempts": 12,
    "occlusion_probability": 0.35,
    "val_fraction": 0.10,
    "num_workers": min(4, os.cpu_count() or 2),

    # Whole-image overlap validation; this is intentionally not a centre crop.
    "val_stride": 384,             # 25% overlap for 512 px windows
    "val_batch_size": 4,
    "validate_every": 2,
    "val_tta": False,              # enable only for final evaluation/inference
    "final_tta": True,              # H/V/both flip ensemble; slower, usually stronger
    "thresholds": [round(x, 2) for x in np.arange(0.20, 0.71, 0.02)],

    # Last epochs gently reward road connectivity without dominating IoU.
    "topology_epochs": 5,
    "topology_weight": 0.10,
    "seed": SEED,
}

OUTPUT_DIR = Path("/kaggle/working/road_segmentation") if ENV == "kaggle" else repo_path / "models" / "road_segmentation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUTPUT_DIR / f"deepglobe_{CONFIG['encoder']}_{CONFIG['image_size']}px_best.pt"
LAST_PATH = OUTPUT_DIR / f"deepglobe_{CONFIG['encoder']}_{CONFIG['image_size']}px_last.pt"
HISTORY_PATH = OUTPUT_DIR / "history.csv"

def paired_images_in(folder):
    folder = Path(folder)
    pairs = []
    for image_path in folder.glob("*_sat.jpg"):
        mask_path = image_path.with_name(image_path.name.replace("_sat.jpg", "_mask.png"))
        if mask_path.exists():
            pairs.append((image_path, mask_path))
    return sorted(pairs)

def find_deepglobe_train_dir():
    if ENV == "kaggle":
        roots = [Path("/kaggle/input")]
    elif ENV == "colab":
        roots = [Path("/content")]
    else:
        roots = [Path("data/raw/deepglobe"), Path("data")]
    candidates = set()
    for root in roots:
        if root.exists():
            for mask_path in root.rglob("*_mask.png"):
                candidates.add(mask_path.parent)
    scored = [(len(paired_images_in(folder)), folder) for folder in candidates]
    scored = [(n, folder) for n, folder in scored if n]
    if not scored:
        raise FileNotFoundError(
            "No DeepGlobe *_sat.jpg / *_mask.png pairs found. Add the dataset with + Add Data."
        )
    return max(scored, key=lambda item: item[0])[1]

DATA_ROOT = find_deepglobe_train_dir()
all_pairs = paired_images_in(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("labelled images:", len(all_pairs))
assert len(all_pairs) >= 20, "Unexpectedly small labelled dataset; verify DATA_ROOT."

def road_fraction(mask_path):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise ValueError(f"Could not read mask: {mask_path}")
    return float((mask > 127).mean())

# Coverage-stratified holdout prevents the validation set from accidentally
# containing mostly empty or mostly road-heavy tiles.
coverages = np.array([road_fraction(mask_path) for _, mask_path in all_pairs])
q = np.quantile(coverages, np.linspace(0, 1, 6))
edges = np.unique(q)
bin_ids = np.digitize(coverages, edges[1:-1], right=True)
rng = np.random.default_rng(SEED)
val_indices = []
for bin_id in np.unique(bin_ids):
    indices = np.flatnonzero(bin_ids == bin_id)
    rng.shuffle(indices)
    take = max(1, round(len(indices) * CONFIG["val_fraction"]))
    val_indices.extend(indices[:take].tolist())
val_indices = set(val_indices)
train_pairs = [pair for i, pair in enumerate(all_pairs) if i not in val_indices]
val_pairs = [pair for i, pair in enumerate(all_pairs) if i in val_indices]
print(f"train: {len(train_pairs)} | validation: {len(val_pairs)} | "
      f"road coverage median: {np.median(coverages):.3%}")


In [ ]:
# 4. Road-aware training dataset and augmentations.
# Keeping crop selection in the dataset avoids wasting many updates on empty tiles.
from torch.utils.data import DataLoader, Dataset
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

class RoadAwareCropDataset(Dataset):
    def __init__(self, pairs, size, crops_per_image, positive_probability,
                 min_positive_fraction, max_attempts, transform):
        self.pairs = pairs
        self.size = size
        self.crops_per_image = crops_per_image
        self.positive_probability = positive_probability
        self.min_positive_fraction = min_positive_fraction
        self.max_attempts = max_attempts
        self.transform = transform

    def __len__(self):
        return len(self.pairs) * self.crops_per_image

    @staticmethod
    def _read(image_path, mask_path):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None:
            raise ValueError(f"Unreadable pair: {image_path}, {mask_path}")
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB), (mask > 127).astype(np.uint8)

    def _crop(self, image, mask):
        h, w = mask.shape
        if h < self.size or w < self.size:
            pad_h, pad_w = max(0, self.size - h), max(0, self.size - w)
            image = cv2.copyMakeBorder(image, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT_101)
            mask = cv2.copyMakeBorder(mask, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=0)
            h, w = mask.shape
        wants_road = np.random.random() < self.positive_probability and mask.any()
        chosen = None
        for _ in range(self.max_attempts):
            y = np.random.randint(0, h - self.size + 1)
            x = np.random.randint(0, w - self.size + 1)
            crop_mask = mask[y:y + self.size, x:x + self.size]
            chosen = (y, x)
            if not wants_road or crop_mask.mean() >= self.min_positive_fraction:
                break
        y, x = chosen
        return image[y:y + self.size, x:x + self.size], mask[y:y + self.size, x:x + self.size]

    def __getitem__(self, idx):
        image, mask = self._read(*self.pairs[idx % len(self.pairs)])
        image, mask = self._crop(image, mask)
        out = self.transform(image=image, mask=mask)
        return out["image"], out["mask"].unsqueeze(0).float()

S = CONFIG["image_size"]
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.18, contrast_limit=0.18, p=0.35),
    A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=8, p=0.20),
    # Blacked-out regions model tree / vehicle / building occlusions while
    # retaining the target road label under each hole.
    A.CoarseDropout(
        num_holes_range=(1, 5),
        hole_height_range=(max(4, S // 32), max(8, S // 9)),
        hole_width_range=(max(4, S // 32), max(8, S // 9)),
        fill=0, fill_mask=None, p=CONFIG["occlusion_probability"],
    ),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

train_ds = RoadAwareCropDataset(
    train_pairs, S, CONFIG["crops_per_image"],
    CONFIG["positive_crop_probability"], CONFIG["min_positive_fraction"],
    CONFIG["max_crop_attempts"], train_transform,
)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

loader_kwargs = {
    "batch_size": CONFIG["batch_size"], "shuffle": True, "drop_last": True,
    "num_workers": CONFIG["num_workers"], "pin_memory": torch.cuda.is_available(),
    "worker_init_fn": seed_worker,
}
if CONFIG["num_workers"] > 0:
    loader_kwargs.update(persistent_workers=True, prefetch_factor=2)
train_loader = DataLoader(train_ds, **loader_kwargs)
print("training crops / epoch:", len(train_ds), "| updates / epoch:",
      math.ceil(len(train_loader) / CONFIG["grad_accum"]))


In [ ]:
# 5. Whole-image, overlap-blended validation. It evaluates every validation
# pixel at native scale and selects a global IoU threshold in one pass.
@torch.inference_mode()
def _predict_batch_prob(net, x, use_tta=False):
    with torch.amp.autocast(device_type="cuda", enabled=(device == "cuda")):
        probs = torch.sigmoid(net(x))
        if use_tta:
            for dims in ((3,), (2,), (2, 3)):
                flipped = torch.flip(x, dims=dims)
                probs += torch.flip(torch.sigmoid(net(flipped)), dims=dims)
            probs /= 4.0
    return probs

def _starts(length, tile, stride):
    if length <= tile:
        return [0]
    starts = list(range(0, length - tile + 1, stride))
    if starts[-1] != length - tile:
        starts.append(length - tile)
    return starts

@torch.inference_mode()
def sliding_window_prob(net, image, tile_size, stride, batch_size, use_tta=False):
    h, w = image.shape[:2]
    hp, wp = max(h, tile_size), max(w, tile_size)
    padded = cv2.copyMakeBorder(image, 0, hp - h, 0, wp - w, cv2.BORDER_REFLECT_101)
    ys, xs = _starts(hp, tile_size, stride), _starts(wp, tile_size, stride)
    yy, xx = np.mgrid[:tile_size, :tile_size]
    # Non-zero Hann weights erase patch seams while still covering borders.
    weight = (np.outer(np.hanning(tile_size), np.hanning(tile_size)) + 0.05).astype(np.float32)
    total = np.zeros((hp, wp), dtype=np.float32)
    norm = np.zeros((hp, wp), dtype=np.float32)
    coords = [(y, x) for y in ys for x in xs]

    for offset in range(0, len(coords), batch_size):
        batch_coords = coords[offset:offset + batch_size]
        tiles = np.stack([padded[y:y + tile_size, x:x + tile_size] for y, x in batch_coords])
        tensor = torch.from_numpy(tiles).float().div_(255.0)
        tensor.sub_(torch.tensor(IMAGENET_MEAN)).div_(torch.tensor(IMAGENET_STD))
        tensor = tensor.permute(0, 3, 1, 2).contiguous(memory_format=torch.channels_last).to(device, non_blocking=True)
        probabilities = _predict_batch_prob(net, tensor, use_tta)[:, 0].float().cpu().numpy()
        for (y, x), probability in zip(batch_coords, probabilities):
            total[y:y + tile_size, x:x + tile_size] += probability * weight
            norm[y:y + tile_size, x:x + tile_size] += weight
    return (total / np.maximum(norm, 1e-6))[:h, :w]

def evaluate_full_images(net, pairs, thresholds, use_tta=False, progress=True):
    intersections = {float(t): 0 for t in thresholds}
    unions = {float(t): 0 for t in thresholds}
    iterator = pairs
    if progress:
        from tqdm.auto import tqdm
        iterator = tqdm(pairs, desc="full-resolution validation", leave=False)
    net.eval()
    for image_path, mask_path in iterator:
        bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        target = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
        image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        probability = sliding_window_prob(
            net, image, CONFIG["image_size"], CONFIG["val_stride"],
            CONFIG["val_batch_size"], use_tta=use_tta,
        )
        for threshold in thresholds:
            prediction = probability >= threshold
            intersections[float(threshold)] += np.logical_and(prediction, target).sum(dtype=np.int64)
            unions[float(threshold)] += np.logical_or(prediction, target).sum(dtype=np.int64)
    ious = {t: intersections[t] / max(unions[t], 1) for t in intersections}
    best_threshold = max(ious, key=ious.get)
    return {
        "iou": float(ious[best_threshold]),
        "threshold": float(best_threshold),
        "iou_by_threshold": ious,
    }


In [ ]:
# 6. Model, discriminative optimizer, AMP scaler, and update-level schedule.
device = "cuda" if torch.cuda.is_available() else "cpu"
if device != "cuda":
    raise RuntimeError("Enable a Kaggle GPU accelerator before training this configuration.")

model = build_model(encoder=CONFIG["encoder"], encoder_weights="imagenet")
model = model.to(device=device, memory_format=torch.channels_last)

def optimizer_groups(net, base_lr, encoder_scale, weight_decay):
    groups = defaultdict(list)
    for name, parameter in net.named_parameters():
        if not parameter.requires_grad:
            continue
        branch = "encoder" if name.startswith("encoder.") else "decoder"
        decay = "no_decay" if parameter.ndim == 1 or name.endswith(".bias") else "decay"
        groups[(branch, decay)].append(parameter)
    result = []
    for (branch, decay), parameters in groups.items():
        result.append({
            "params": parameters,
            "lr": base_lr * (encoder_scale if branch == "encoder" else 1.0),
            "weight_decay": 0.0 if decay == "no_decay" else weight_decay,
        })
    return result

optimizer = torch.optim.AdamW(
    optimizer_groups(model, CONFIG["base_lr"], CONFIG["encoder_lr_scale"], CONFIG["weight_decay"]),
    betas=(0.9, 0.999), eps=1e-8,
)
loss_fn = DiceBCELoss(bce_weight=0.45, cldice_weight=0.0)
scaler = torch.amp.GradScaler("cuda", enabled=True)

updates_per_epoch = math.ceil(len(train_loader) / CONFIG["grad_accum"])
total_updates = updates_per_epoch * CONFIG["epochs"]
warmup_updates = updates_per_epoch * CONFIG["warmup_epochs"]

def lr_factor(update):
    if update < warmup_updates:
        return max(1e-3, (update + 1) / max(1, warmup_updates))
    phase = (update - warmup_updates) / max(1, total_updates - warmup_updates)
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, phase)))
    return CONFIG["min_lr_ratio"] + (1.0 - CONFIG["min_lr_ratio"]) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_factor)
print("model:", CONFIG["encoder"], "| parameters (M):",
      round(sum(p.numel() for p in model.parameters()) / 1e6, 2))
print("effective batch:", CONFIG["batch_size"] * CONFIG["grad_accum"],
      "| total optimizer updates:", total_updates)


In [ ]:
# 7. Train, validate at full native resolution, and persist both best and resumable state.
import csv
from tqdm.auto import tqdm

def unwrap(net):
    return net.module if hasattr(net, "module") else net

def save_state(path, epoch, best_iou, threshold, metrics, include_optimizer):
    payload = {
        "state_dict": unwrap(model).state_dict(),
        "meta": {
            "encoder": CONFIG["encoder"],
            "image_size": CONFIG["image_size"],
            "threshold": float(threshold),
            "best_full_resolution_iou": float(best_iou),
            "epoch": int(epoch),
            "config": CONFIG.copy(),
            "metrics": metrics,
        },
    }
    if include_optimizer:
        payload.update({
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "epoch": int(epoch), "best_iou": float(best_iou),
        })
    torch.save(payload, path)

# Resume is opt-in: set RESUME_PATH to LAST_PATH after a timed-out session.
RESUME_PATH = None
start_epoch, best_iou, best_threshold = 1, -1.0, 0.5
if RESUME_PATH:
    # RESUME_PATH is a checkpoint written by this notebook. Explicitly opt
    # into loading its trusted training metadata on PyTorch 2.6+.
    resumed = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    unwrap(model).load_state_dict(resumed["state_dict"])
    optimizer.load_state_dict(resumed["optimizer"])
    scheduler.load_state_dict(resumed["scheduler"])
    scaler.load_state_dict(resumed["scaler"])
    start_epoch = int(resumed["epoch"]) + 1
    best_iou = float(resumed.get("best_iou", -1.0))
    best_threshold = float(resumed.get("meta", {}).get("threshold", 0.5))
    print(f"resumed from epoch {start_epoch - 1}; best IoU {best_iou:.5f}")

history = []
with open(HISTORY_PATH, "a", newline="") as history_file:
    writer = csv.DictWriter(history_file, fieldnames=[
        "epoch", "train_loss", "lr", "topology_weight", "val_iou", "threshold"
    ])
    if history_file.tell() == 0:
        writer.writeheader()

    for epoch in range(start_epoch, CONFIG["epochs"] + 1):
        # Switch on a small topology term only after pixel-level segmentation is stable.
        loss_fn.cldice_weight = (
            CONFIG["topology_weight"]
            if epoch > CONFIG["epochs"] - CONFIG["topology_epochs"] else 0.0
        )
        model.train()
        optimizer.zero_grad(set_to_none=True)
        running_loss, batches = 0.0, 0
        progress = tqdm(train_loader, desc=f"epoch {epoch:02d}/{CONFIG['epochs']}", leave=False)

        for batch_idx, (images, masks) in enumerate(progress):
            # The final optimizer group may be shorter than grad_accum;
            # scale every member of that group by the same true group size.
            group_start = batch_idx - (batch_idx % CONFIG["grad_accum"])
            group_size = min(CONFIG["grad_accum"], len(train_loader) - group_start)
            images = images.to(device, non_blocking=True).contiguous(memory_format=torch.channels_last)
            masks = masks.to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", enabled=True):
                loss = loss_fn(model(images), masks)
                scaled_loss = loss / group_size
            scaler.scale(scaled_loss).backward()
            running_loss += float(loss.detach())
            batches += 1

            is_update = ((batch_idx + 1) % CONFIG["grad_accum"] == 0) or (batch_idx + 1 == len(train_loader))
            if is_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            progress.set_postfix(loss=f"{running_loss / max(batches, 1):.4f}",
                                 lr=f"{optimizer.param_groups[-1]['lr']:.2e}")

        row = {
            "epoch": epoch,
            "train_loss": running_loss / max(batches, 1),
            "lr": optimizer.param_groups[-1]["lr"],
            "topology_weight": loss_fn.cldice_weight,
            "val_iou": "",
            "threshold": "",
        }
        validate_now = epoch == 1 or epoch % CONFIG["validate_every"] == 0 or epoch == CONFIG["epochs"]
        metrics = {"validation_skipped": True}
        if validate_now:
            metrics = evaluate_full_images(model, val_pairs, CONFIG["thresholds"], use_tta=CONFIG["val_tta"])
            row["val_iou"] = metrics["iou"]
            row["threshold"] = metrics["threshold"]
            print(f"epoch {epoch:02d} | loss {row['train_loss']:.4f} | full-val IoU {metrics['iou']:.5f} "
                  f"@ threshold {metrics['threshold']:.2f} | lr {row['lr']:.2e}")
            if metrics["iou"] > best_iou:
                best_iou, best_threshold = metrics["iou"], metrics["threshold"]
                save_state(BEST_PATH, epoch, best_iou, best_threshold, metrics, include_optimizer=False)
                print("  saved new best checkpoint ->", BEST_PATH)
        else:
            print(f"epoch {epoch:02d} | loss {row['train_loss']:.4f} | validation deferred | lr {row['lr']:.2e}")

        writer.writerow(row)
        history.append(row)
        history_file.flush()
        save_state(LAST_PATH, epoch, best_iou, best_threshold, metrics, include_optimizer=True)
        gc.collect()
        torch.cuda.empty_cache()

assert BEST_PATH.exists(), "No best checkpoint was written. Inspect the training log above."
print(f"best non-TTA full-resolution IoU: {best_iou:.5f} @ threshold {best_threshold:.2f}")


In [ ]:
# 8. Restore the actual best weights, then run the final (optional) flip-TTA evaluation.
# This fixes the common failure mode where later, worse epoch weights get evaluated/saved.
# BEST_PATH was written by this notebook. Its metadata includes NumPy scalar
# values, so PyTorch 2.6+ needs this explicit trusted-checkpoint setting.
best_payload = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
# Keep this cell independently runnable after a recovered checkpoint: unwrap
# is defined in the training cell, which a recovery workflow intentionally skips.
model.load_state_dict(best_payload["state_dict"])
model = model.to(device=device, memory_format=torch.channels_last).eval()

final_metrics = evaluate_full_images(
    model, val_pairs, CONFIG["thresholds"], use_tta=CONFIG["final_tta"]
)
final_threshold = final_metrics["threshold"]
print("final full-resolution", "flip-TTA" if CONFIG["final_tta"] else "single-view",
      f"IoU: {final_metrics['iou']:.5f} @ threshold {final_threshold:.2f}")

# Store the inference threshold with the same weights users will export.
best_payload["meta"].update({
    "threshold": float(final_threshold),
    "final_tta": bool(CONFIG["final_tta"]),
    "final_full_resolution_iou": float(final_metrics["iou"]),
    "final_metrics": final_metrics,
})
torch.save(best_payload, BEST_PATH)


In [ ]:
# 9. Occlusion-recall check on a reproducible validation subset.
# It measures only road pixels deliberately hidden by synthetic obstacles.
def occlude_image(image, rng, max_holes=5):
    image = image.copy()
    h, w = image.shape[:2]
    occlusion = np.zeros((h, w), dtype=np.uint8)
    for _ in range(rng.integers(1, max_holes + 1)):
        hh = int(rng.integers(max(8, S // 16), max(9, S // 7)))
        ww = int(rng.integers(max(8, S // 16), max(9, S // 7)))
        y = int(rng.integers(0, max(1, h - hh + 1)))
        x = int(rng.integers(0, max(1, w - ww + 1)))
        image[y:y + hh, x:x + ww] = 0
        occlusion[y:y + hh, x:x + ww] = 1
    return image, occlusion

occ_rng = np.random.default_rng(SEED + 99)
hidden_road, recovered_road = 0, 0
OCCLUSION_EVAL_IMAGES = min(100, len(val_pairs))
for image_path, mask_path in val_pairs[:OCCLUSION_EVAL_IMAGES]:
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    target = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
    occluded, occ_mask = occlude_image(image, occ_rng)
    probability = sliding_window_prob(
        model, occluded, CONFIG["image_size"], CONFIG["val_stride"],
        CONFIG["val_batch_size"], use_tta=CONFIG["final_tta"],
    )
    hidden = target & occ_mask.astype(bool)
    hidden_road += hidden.sum(dtype=np.int64)
    recovered_road += ((probability >= final_threshold) & hidden).sum(dtype=np.int64)

occ_recall = recovered_road / max(hidden_road, 1)
print(f"Occlusion-Recall ({OCCLUSION_EVAL_IMAGES} validation images): {occ_recall:.5f}")


In [ ]:
# 10. Visual sanity check and explicit export paths.
import matplotlib.pyplot as plt

image_path, mask_path = val_pairs[0]
image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
target = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 127
probability = sliding_window_prob(
    model, image, CONFIG["image_size"], CONFIG["val_stride"],
    CONFIG["val_batch_size"], use_tta=CONFIG["final_tta"],
)
prediction = probability >= final_threshold

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(image); axes[0].set_title("satellite image")
axes[1].imshow(target, cmap="gray"); axes[1].set_title("ground truth")
axes[2].imshow(probability, cmap="magma", vmin=0, vmax=1); axes[2].set_title("road probability")
axes[3].imshow(prediction, cmap="gray"); axes[3].set_title(f"prediction @ {final_threshold:.2f}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Best deployable checkpoint:", BEST_PATH, "| exists:", BEST_PATH.exists())
print("Resumable checkpoint:", LAST_PATH, "| exists:", LAST_PATH.exists())
print("Training history:", HISTORY_PATH, "| exists:", HISTORY_PATH.exists())
print("\nKaggle: use Save Version to retain /kaggle/working/road_segmentation as notebook output.")


## Inference notes

The best checkpoint is compatible with the project loader because it contains the
canonical `state_dict` plus `meta`. The stored `meta['threshold']` is the selected
full-resolution threshold. For production-sized AOIs, retain the same 512 px sliding
window / 384 px stride policy used in validation; enable flip-TTA only when accuracy
matters more than inference time.

The validation score is a quality-control signal, not a public leaderboard result.
DeepGlobe tiles can be spatially related, so test on geographically separate imagery
before relying on a routing-quality claim.
